# Imports

In [77]:
import ultralytics
import cv2
import numpy as np 
import matplotlib.pyplot as plt 
import pickle 
import pandas as pd  
import torch 
import torch.nn as nn 
import torch.optim as optim 
import torchvision 
import torchvision.transforms as transforms 
import torchvision.datasets as datasets 
import torchvision.models as models 

DATASET_PATH="/home/nyan/FineBio"


In [78]:
model=ultralytics.YOLO("yolo26n.pt")


# FineBio + YOLO-26: multi-view detection and atomic/step recognition

In this section we:
- align with the FineBio dataset on disk,
- run YOLO-26 for object / manipulated-object detection,
- build symbolic features over time,
- and set up a simple temporal model for atomic operations and steps.


In [79]:
from pathlib import Path
import json
from dataclasses import dataclass
from typing import List, Dict, Optional, Tuple

DATASET_PATH = Path("/home/nyan/FineBio")
ANNOTATIONS_DIR = DATASET_PATH / "01 annotations"
COCO_ZIP = ANNOTATIONS_DIR / "finebio_coco_annotations.zip"
ACTION_ZIP = ANNOTATIONS_DIR / "finebio_action_annotations.zip"

IMAGES_ROOT = DATASET_PATH / "12 finebio_object_detection_images" / "finebio_object_detection_images"

print("Dataset root:", DATASET_PATH)
print("Images root:", IMAGES_ROOT)
print("Annotations dir:", ANNOTATIONS_DIR)


Dataset root: /home/nyan/FineBio
Images root: /home/nyan/FineBio/12 finebio_object_detection_images/finebio_object_detection_images
Annotations dir: /home/nyan/FineBio/01 annotations


In [80]:
import zipfile

# Helper to ensure COCO annotations are extracted next to the zip
COCO_DIR = ANNOTATIONS_DIR / "finebio_coco_annotations"
ACTION_DIR = ANNOTATIONS_DIR / "finebio_action_annotations"

if not COCO_DIR.exists() and COCO_ZIP.exists():
    with zipfile.ZipFile(COCO_ZIP, "r") as zf:
        zf.extractall(ANNOTATIONS_DIR)
        print("Extracted COCO annotations to", COCO_DIR)

if not ACTION_DIR.exists() and ACTION_ZIP.exists():
    with zipfile.ZipFile(ACTION_ZIP, "r") as zf:
        zf.extractall(ANNOTATIONS_DIR)
        print("Extracted action annotations to", ACTION_DIR)

print("COCO_DIR exists:", COCO_DIR.exists())
print("ACTION_DIR exists:", ACTION_DIR.exists())


COCO_DIR exists: True
ACTION_DIR exists: True


In [81]:
# Load one of the FineBio COCO-style annotation files (joint train+val as default)
COCO_JSON = COCO_DIR / "v1_trainval_joint.json"

with open(COCO_JSON, "r") as f:
    coco_data = json.load(f)

categories = {c["id"]: c["name"] for c in coco_data["categories"]}
cat_name_to_id = {v: k for k, v in categories.items()}

print("Loaded COCO annotations from", COCO_JSON)
print("Number of categories:", len(categories))
print("Sample categories:", list(categories.values())[:10])

# Build quick image-id -> file-name mapping
image_id_to_info = {img["id"]: img for img in coco_data["images"]}
file_name_to_image_id = {img["file_name"]: img["id"] for img in coco_data["images"]}

print("Number of images in COCO JSON:", len(image_id_to_info))


Loaded COCO annotations from /home/nyan/FineBio/01 annotations/finebio_coco_annotations/v1_trainval_joint.json
Number of categories: 35
Sample categories: ['left_hand', 'right_hand', 'blue_pipette', 'yellow_pipette', 'red_pipette', '8_channel_pipette', 'blue_tip', 'yellow_tip', 'red_tip', '8_channel_tip']
Number of images in COCO JSON: 1632


In [82]:
# Parse action (atomic operation + step) annotations for a given experiment instance

@dataclass
class FineBioAction:
    start_sec: float
    end_sec: float
    task: str  # coarse step-level task (e.g., remove_culture_medium)
    hand_side: Optional[str]
    verb: Optional[str]
    manipulated_object: Optional[str]
    affected_object: Optional[str]


def load_actions_for_instance(instance_id: str) -> List[FineBioAction]:
    """Load atomic operations + step-like tasks for one experiment instance.

    Args:
        instance_id: e.g. "P03_02_01" corresponding to FineBio naming.
    """
    txt_path = ACTION_DIR / f"{instance_id}.txt"
    actions: List[FineBioAction] = []
    if not txt_path.exists():
        print("No action annotation file:", txt_path)
        return actions

    with open(txt_path, "r") as f:
        header = f.readline().strip().split(",")
        # Expect: start_sec,end_sec,task,hand_side,verb,manipulated_object,affected_object
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split(",")
            if len(parts) < 2:
                continue
            # Pad to length 7
            while len(parts) < 7:
                parts.append("")
            start_sec, end_sec = float(parts[0]), float(parts[1])
            task, hand_side, verb, manipulated_object, affected_object = parts[2:7]
            actions.append(
                FineBioAction(
                    start_sec=start_sec,
                    end_sec=end_sec,
                    task=task or "",
                    hand_side=hand_side or None,
                    verb=verb or None,
                    manipulated_object=manipulated_object or None,
                    affected_object=affected_object or None,
                )
            )
    print(f"Loaded {len(actions)} actions for {instance_id}")
    return actions


# Quick sanity check on one instance from the paper example
_example_instance = "P03_02_01"
_example_actions = load_actions_for_instance(_example_instance)
_example_actions[:5]


Loaded 169 actions for P03_02_01


[FineBioAction(start_sec=1.0, end_sec=24.92, task='remove_culture_medium', hand_side=None, verb=None, manipulated_object=None, affected_object=None),
 FineBioAction(start_sec=24.96, end_sec=39.85, task='add_pbs', hand_side=None, verb=None, manipulated_object=None, affected_object=None),
 FineBioAction(start_sec=39.87, end_sec=44.9, task='shake_plate', hand_side=None, verb=None, manipulated_object=None, affected_object=None),
 FineBioAction(start_sec=44.97, end_sec=56.07, task='aspirate_pbs', hand_side=None, verb=None, manipulated_object=None, affected_object=None),
 FineBioAction(start_sec=56.16, end_sec=69.16, task='add_pbs', hand_side=None, verb=None, manipulated_object=None, affected_object=None)]

In [83]:
# YOLO-26 based object detection and symbolic feature extraction

from collections import Counter

# model is already created above as `model = ultralytics.YOLO("yolo26n.pt")`
# We assume it has been (or will be) fine-tuned on FineBio categories so that
# `model.names` is aligned with object classes like blue_pipette, blue_tip, etc.

model_names: Dict[int, str] = model.names  # id -> name


def yolo_features_for_image(img_path: Path,
                            conf: float = 0.25) -> Tuple[Counter, List[Dict]]:
    """Run YOLO-26 on one image and return
    - a Counter over predicted class names (symbolic feature),
    - the raw detection records (for debugging / visualization).
    """
    results = model(str(img_path), conf=conf, verbose=False)
    if not results:
        return Counter(), []
    res = results[0]
    detections: List[Dict] = []
    counts: Counter = Counter()

    if res.boxes is None:
        return counts, detections

    for b in res.boxes:
        cls_id = int(b.cls.item())
        score = float(b.conf.item())
        name = model_names.get(cls_id, str(cls_id))
        counts[name] += 1
        x1, y1, x2, y2 = map(float, b.xyxy[0].tolist())
        detections.append({
            "class_id": cls_id,
            "class_name": name,
            "score": score,
            "bbox": [x1, y1, x2, y2],
        })

    return counts, detections


# Example on one FineBio object-detection image
from itertools import islice

sample_images = list(islice(IMAGES_ROOT.glob("*.jpg"), 5))
print("Found", len(sample_images), "sample images")
if sample_images:
    c, dets = yolo_features_for_image(sample_images[0])
    print("Sample image:", sample_images[0].name)
    print("Symbolic feature (counts per class):", c)
    print("Num detections:", len(dets))


Found 5 sample images
Sample image: P28_05_01_000407.jpg
Symbolic feature (counts per class): Counter({'keyboard': 1, 'person': 1, 'book': 1})
Num detections: 3


In [84]:
# From per-frame YOLO features to a temporal sequence for atomic operations / steps

import math


def assign_action_label(time_sec: float, actions: List[FineBioAction]) -> Optional[FineBioAction]:
    """Find which action interval (if any) covers this timestamp."""
    for a in actions:
        if a.start_sec <= time_sec <= a.end_sec:
            return a
    return None


def resolve_video_paths(instance_id: str, view: str = "tpv") -> List[Path]:
    """Return all matching video paths for a given instance id and view.

    - TPV videos live in 05/06/08 .../finebio_videos and use Pxx_xx_xx_T1..T5.mp4
    - FPV videos live in 09/11 .../finebio_videos and 04 (test),
      and can be either Pxx_xx_xx.mp4 or Pxx_xx_xx_T1..T5.mp4.
    """
    paths: List[Path] = []

    if view == "tpv":
        tpv_dirs = [
            DATASET_PATH / "05 finebio_videos_tpv_train" / "finebio_videos",
            DATASET_PATH / "06 finebio_videos_tpv_valid" / "finebio_videos",
            DATASET_PATH / "08 finebio_videos_tpv_test" / "finebio_videos",
        ]
        trials = ["T1", "T2", "T3", "T4", "T5"]
        for d in tpv_dirs:
            for t in trials:
                cand = d / f"{instance_id}_{t}.mp4"
                if cand.exists():
                    paths.append(cand)
    elif view == "fpv":
        fpv_dirs = [
            DATASET_PATH / "09 finebio_videos_fpv_train" / "finebio_videos",
            DATASET_PATH / "11 finebio_videos_fpv_valid" / "finebio_videos",
            DATASET_PATH / "04 finebio_videos_fpv_test",
        ]
        trials = ["T1", "T2", "T3", "T4", "T5"]
        for d in fpv_dirs:
            # single-clip version
            cand_single = d / f"{instance_id}.mp4"
            if cand_single.exists():
                paths.append(cand_single)
            # trial versions
            for t in trials:
                cand = d / f"{instance_id}_{t}.mp4"
                if cand.exists():
                    paths.append(cand)
    else:
        raise ValueError(f"Unknown view: {view}")

    return paths


def build_sequence_for_instance(instance_id: str,
                                fps: float = 1.0,
                                view: str = "tpv",
                                max_frames: Optional[int] = None,
                                conf: float = 0.25) -> List[Dict]:
    """Construct a time sequence of symbolic YOLO features + labels for one instance.

    Returns a list of dicts, each containing:
        - time_sec
        - yolo_class_counts (dict[class_name -> count])
        - detections (raw YOLO detections)
        - action (FineBioAction or None)
    """
    from collections import Counter

    actions = load_actions_for_instance(instance_id)
    if not actions:
        return []

    video_paths = resolve_video_paths(instance_id, view=view)
    if not video_paths:
        print(f"No video found for {instance_id} (view={view})")
        return []

    # For now, just use the first matching video
    video_path = video_paths[0]
    print("Using video:", video_path)

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        print("Failed to open video", video_path)
        return []

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    video_fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    duration_sec = total_frames / video_fps if video_fps > 0 else 0.0
    print(f"Video FPS={video_fps:.2f}, frames={total_frames}, duration={duration_sec:.2f}s")

    frame_interval = max(int(round(video_fps / fps)), 1)
    seq: List[Dict] = []

    frame_idx = 0
    sampled = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        if frame_idx % frame_interval == 0:
            time_sec = frame_idx / video_fps

            # Run YOLO directly on the numpy frame (OpenCV BGR is fine)
            results = model(frame, conf=conf, verbose=False)
            counts = Counter()
            dets: List[Dict] = []
            if results:
                res = results[0]
                if res.boxes is not None:
                    for b in res.boxes:
                        cls_id = int(b.cls.item())
                        score = float(b.conf.item())
                        name = model_names.get(cls_id, str(cls_id))
                        counts[name] += 1
                        x1, y1, x2, y2 = map(float, b.xyxy[0].tolist())
                        dets.append({
                            "class_id": cls_id,
                            "class_name": name,
                            "score": score,
                            "bbox": [x1, y1, x2, y2],
                        })

            action = assign_action_label(time_sec, actions)
            seq.append({
                "time_sec": time_sec,
                "yolo_class_counts": dict(counts),
                "detections": dets,
                "action": action,
            })

            sampled += 1
            if max_frames is not None and sampled >= max_frames:
                break

        frame_idx += 1

    cap.release()
    print(f"Built sequence of length {len(seq)} for {instance_id} ({view})")
    return seq


# Skeleton for a small temporal model (you can refine/extend this later)

class SimpleGRUHead(nn.Module):
    """A lightweight GRU-based classifier over YOLO-derived symbolic features.

    You would feed sequences of per-frame feature vectors (e.g. class-counts
    embedded into a fixed-size vector) and train it to predict atomic
    operations or steps.
    """

    def __init__(self, input_dim: int, hidden_dim: int, num_classes: int, num_layers: int = 1):
        super().__init__()
        self.gru = nn.GRU(input_dim, hidden_dim, num_layers=num_layers, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, x):
        # x: (B, T, D)
        h, _ = self.gru(x)
        logits = self.fc(h)  # (B, T, num_classes)
        return logits


### What you can run when you’re tired

- Run all cells up to the bottom: this wires FineBio, loads annotations, and creates the YOLO-26 model.
- Call `build_sequence_for_instance("P03_02_01", fps=1.0, view="fpv", max_frames=30)` to get a ready-made sequence with YOLO counts + atomic/step labels.
- You can then plug that into any temporal model (like `SimpleGRUHead`) when you feel like training; the heavy wiring work is already done.


# Use Ultralytics Yolo 26


# Train YOLO-26 on FineBio object detection (COCO annotations)

We first fine-tune YOLO-26 on the FineBio COCO annotations so it learns lab objects
(blue_pipette, tips, tubes, racks, etc.) instead of generic COCO classes (person, laptop, bottle).

In [85]:
# Prepare YOLO-format dataset from FineBio COCO JSONs (train/val)

import os
from collections import defaultdict

FINEBIO_YOLO_ROOT = Path("/home/nyan/k-project/finebio_yolo_joint")
FINEBIO_YOLO_ROOT.mkdir(parents=True, exist_ok=True)

IMAGES_SRC = IMAGES_ROOT  # original FineBio object detection images

TRAIN_JSON = COCO_DIR / "v1_train_joint.json"
VAL_JSON = COCO_DIR / "v1_valid_joint.json"

images_train_dir = FINEBIO_YOLO_ROOT / "images" / "train"
images_val_dir = FINEBIO_YOLO_ROOT / "images" / "val"
labels_train_dir = FINEBIO_YOLO_ROOT / "labels" / "train"
labels_val_dir = FINEBIO_YOLO_ROOT / "labels" / "val"

for d in [images_train_dir, images_val_dir, labels_train_dir, labels_val_dir]:
    d.mkdir(parents=True, exist_ok=True)

# Load categories from train JSON and define class mapping (COCO id -> 0-based index)
with open(TRAIN_JSON, "r") as f:
    train_coco = json.load(f)

categories_joint = sorted(train_coco["categories"], key=lambda c: c["id"])
cat_id_to_idx = {c["id"]: i for i, c in enumerate(categories_joint)}
finebio_class_names = [c["name"] for c in categories_joint]
print("Number of FineBio classes:", len(finebio_class_names))
print("First few classes:", finebio_class_names[:10])


def prepare_split(coco_json_path: Path, images_dest: Path, labels_dest: Path):
    print("Preparing split from", coco_json_path)
    with open(coco_json_path, "r") as f:
        data = json.load(f)

    images = {img["id"]: img for img in data["images"]}
    anns_by_image: Dict[int, list] = defaultdict(list)
    for ann in data["annotations"]:
        anns_by_image[ann["image_id"]].append(ann)

    num_images = len(images)
    num_anns = len(data["annotations"])
    print(f"  images={num_images}, annotations={num_anns}")

    for img_id, img in images.items():
        fname = img["file_name"]
        width, height = img["width"], img["height"]
        src_img = IMAGES_SRC / fname
        if not src_img.exists():
            # Some images may be missing; skip gracefully
            continue

        dst_img = images_dest / fname
        if not dst_img.exists():
            try:
                os.symlink(src_img, dst_img)
            except OSError:
                # Fallback to copy if symlink is not permitted
                import shutil
                shutil.copy2(src_img, dst_img)

        anns = anns_by_image.get(img_id, [])
        if not anns:
            # No annotations -> background image; YOLO is fine with missing txt
            continue

        label_path = labels_dest / (Path(fname).stem + ".txt")
        with open(label_path, "w") as lf:
            for ann in anns:
                cat_id = ann["category_id"]
                if cat_id not in cat_id_to_idx:
                    continue
                cls_idx = cat_id_to_idx[cat_id]
                x, y, w, h = ann["bbox"]  # COCO: top-left x,y,width,height
                cx = (x + w / 2.0) / width
                cy = (y + h / 2.0) / height
                nw = w / width
                nh = h / height
                lf.write(f"{cls_idx} {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}\n")


prepare_split(TRAIN_JSON, images_train_dir, labels_train_dir)
prepare_split(VAL_JSON, images_val_dir, labels_val_dir)

print("Finished preparing YOLO-format dataset at", FINEBIO_YOLO_ROOT)

Number of FineBio classes: 35
First few classes: ['left_hand', 'right_hand', 'blue_pipette', 'yellow_pipette', 'red_pipette', '8_channel_pipette', 'blue_tip', 'yellow_tip', 'red_tip', '8_channel_tip']
Preparing split from /home/nyan/FineBio/01 annotations/finebio_coco_annotations/v1_train_joint.json
  images=1394, annotations=50886
Preparing split from /home/nyan/FineBio/01 annotations/finebio_coco_annotations/v1_valid_joint.json
  images=238, annotations=8867
Finished preparing YOLO-format dataset at /home/nyan/k-project/finebio_yolo_joint


In [86]:
# Create a Ultralytics data.yaml for FineBio joint object detection

data_yaml_path = FINEBIO_YOLO_ROOT / "finebio_joint.yaml"

names_lines = "\n".join([f"  {i}: {name}" for i, name in enumerate(finebio_class_names)])

yaml_text = f"""path: {FINEBIO_YOLO_ROOT}
train: images/train
val: images/val
names:
{names_lines}
"""

with open(data_yaml_path, "w") as f:
    f.write(yaml_text)

print("Wrote data.yaml to", data_yaml_path)
print("\nPreview:\n", yaml_text)

Wrote data.yaml to /home/nyan/k-project/finebio_yolo_joint/finebio_joint.yaml

Preview:
 path: /home/nyan/k-project/finebio_yolo_joint
train: images/train
val: images/val
names:
  0: left_hand
  1: right_hand
  2: blue_pipette
  3: yellow_pipette
  4: red_pipette
  5: 8_channel_pipette
  6: blue_tip
  7: yellow_tip
  8: red_tip
  9: 8_channel_tip
  10: blue_tip_rack
  11: yellow_tip_rack
  12: red_tip_rack
  13: 8_channel_tip_rack
  14: 50ml_tube
  15: 15ml_tube
  16: micro_tube
  17: 8_tube_stripes
  18: 8_tube_stripes_lid
  19: 50ml_tube_rack
  20: 15ml_tube_rack
  21: micro_tube_rack
  22: 8_tube_stripes_rack
  23: 8_tube_stripes_rack_lid
  24: cell_culture_plate
  25: cell_culture_plate_lid
  26: trash_can
  27: centrifuge
  28: vortex_mixer
  29: magnetic_rack
  30: pcr_machine
  31: tube_with_spin_column
  32: spin_column
  33: tube_without_lid
  34: pen



In [87]:
# # Fine-tune YOLO-26 on FineBio joint object detection

# finebio_data_yaml = str(data_yaml_path)

# # Start from the generic YOLO-26 weights you already loaded
# # (model variable defined above as ultralytics.YOLO("yolo26n.pt"))

# print("Training YOLO-26 on FineBio... this may take a while.")

# train_results = model.train(
#     data=finebio_data_yaml,
#     epochs=30,
#     imgsz=640,
#     batch=16,
#     project="runs/finebio_yolo26",
#     name="yolo26n_joint",
#     patience=5,
#     device=0,  # use GPU 0 explicitly
# )

# print("Training finished.")

# from pathlib import Path
# save_dir = Path(model.trainer.save_dir)  # e.g. runs/finebio_yolo26/yolo26n_joint
# best_weights = save_dir / "weights" / "best.pt"
# print("Best weights saved at:", best_weights)

In [88]:
# Reload YOLO-26 with FineBio-trained weights and sanity-check detections

from pathlib import Path

# Hard-code the latest FineBio-trained checkpoint so you never have to retrain just to get the path
best_weights = Path('/home/nyan/k-project/runs/detect/runs/finebio_yolo26/yolo26n_joint3/weights/best.pt')
print('Using FineBio weights:', best_weights)

# Reload model from the best FineBio weights
model = ultralytics.YOLO(str(best_weights))
model_names = model.names
print("Loaded FineBio-trained YOLO-26. Number of classes:", len(model_names))
print("First few classes:", list(model_names.values())[:10])

# Quick check on one FineBio object-detection image
from itertools import islice

sample_images = list(islice(IMAGES_ROOT.glob("*.jpg"), 5))
print("Found", len(sample_images), "sample images for sanity check")
if sample_images:
    c, dets = yolo_features_for_image(sample_images[0], conf=0.25)
    print("Sample image:", sample_images[0].name)
    print("Symbolic feature (counts per class):", c)
    print("Num detections:", len(dets))


Using FineBio weights: /home/nyan/k-project/runs/detect/runs/finebio_yolo26/yolo26n_joint3/weights/best.pt
Loaded FineBio-trained YOLO-26. Number of classes: 35
First few classes: ['left_hand', 'right_hand', 'blue_pipette', 'yellow_pipette', 'red_pipette', '8_channel_pipette', 'blue_tip', 'yellow_tip', 'red_tip', '8_channel_tip']
Found 5 sample images for sanity check
Sample image: P28_05_01_000407.jpg
Symbolic feature (counts per class): Counter({'8_channel_tip_rack': 4, 'vortex_mixer': 2, '50ml_tube_rack': 2, 'centrifuge': 2, 'micro_tube': 2, '8_tube_stripes': 2, 'trash_can': 1, 'magnetic_rack': 1, 'cell_culture_plate': 1, 'pcr_machine': 1, 'right_hand': 1, '8_tube_stripes_rack_lid': 1, 'left_hand': 1, 'red_tip_rack': 1, '8_tube_stripes_rack': 1, '8_channel_pipette': 1, 'micro_tube_rack': 1, 'yellow_tip_rack': 1})
Num detections: 26


In [89]:
# Quick demo: build a short FPV sequence and inspect first few entries

example_seq = build_sequence_for_instance("P03_02_01", fps=1.0, view="fpv", max_frames=10, conf=0.25)
print("Sequence length:", len(example_seq))
for step in example_seq[:5]:
    a = step["action"]
    print(f"t={step['time_sec']:.2f}s, counts={step['yolo_class_counts']}, "
          f"task={a.task if a else None}, verb={a.verb if a else None}, obj={a.manipulated_object if a else None}")


Loaded 169 actions for P03_02_01
Using video: /home/nyan/FineBio/04 finebio_videos_fpv_test/P03_02_01.mp4
Video FPS=29.97, frames=4613, duration=153.92s
Built sequence of length 10 for P03_02_01 (fpv)
Sequence length: 10
t=0.00s, counts={'centrifuge': 1, 'blue_tip_rack': 1, 'yellow_tip_rack': 1, 'cell_culture_plate': 1, 'micro_tube': 5, 'micro_tube_rack': 1, '50ml_tube_rack': 2, '8_channel_tip_rack': 1, '50ml_tube': 1, 'trash_can': 1, 'red_pipette': 1}, task=None, verb=None, obj=None
t=1.00s, counts={'magnetic_rack': 1, 'blue_tip_rack': 1, 'centrifuge': 1, 'trash_can': 1, '8_channel_tip_rack': 2, 'red_tip_rack': 1, 'cell_culture_plate': 1, '8_channel_pipette': 2, 'left_hand': 2, 'micro_tube': 8, 'pcr_machine': 2, '50ml_tube_rack': 3, 'yellow_tip_rack': 2, '8_tube_stripes_rack': 3, 'right_hand': 1}, task=remove_culture_medium, verb=None, obj=None
t=2.00s, counts={'vortex_mixer': 1, 'cell_culture_plate': 1, 'magnetic_rack': 1, 'centrifuge': 1, '8_tube_stripes_rack': 1, '50ml_tube_rack': 

# Multi-view YOLO-26 + atomic operation / step sequences

Now that YOLO-26 is fine-tuned on FineBio, we build **multi-view** sequences (FPV + TPV)
with fused object detections aligned to atomic operations and steps.

In [90]:
# Helper: run YOLO-26 on a raw frame (numpy array) and return class-counts + detections

from collections import Counter


def yolo_counts_from_frame(frame, conf: float = 0.25):
    """Run YOLO on a single BGR frame and return (counts, detections)."""
    results = model(frame, conf=conf, verbose=False)
    counts = Counter()
    dets: List[Dict] = []
    if results:
        res = results[0]
        if res.boxes is not None:
            for b in res.boxes:
                cls_id = int(b.cls.item())
                score = float(b.conf.item())
                name = model_names.get(cls_id, str(cls_id))
                counts[name] += 1
                x1, y1, x2, y2 = map(float, b.xyxy[0].tolist())
                dets.append({
                    "class_id": cls_id,
                    "class_name": name,
                    "score": score,
                    "bbox": [x1, y1, x2, y2],
                })
    return counts, dets


def build_multiview_sequence_for_instance(instance_id: str,
                                          fps: float = 1.0,
                                          max_frames: Optional[int] = None,
                                          conf: float = 0.25) -> List[Dict]:
    """FPV + TPV fused YOLO features + labels for one experiment instance.

    For each sampled timestamp, we:
    - grab the nearest FPV frame
    - grab the nearest TPV frame (first available TPV camera)
    - run YOLO on both
    - fuse counts across views
    - attach the aligned FineBioAction (atomic/step).
    """
    actions = load_actions_for_instance(instance_id)
    if not actions:
        return []

    fpv_paths = resolve_video_paths(instance_id, view="fpv")
    tpv_paths = resolve_video_paths(instance_id, view="tpv")
    if not fpv_paths or not tpv_paths:
        print("Missing FPV or TPV video for", instance_id)
        return []

    fpv_path = fpv_paths[0]
    tpv_path = tpv_paths[0]
    print("Using FPV:", fpv_path)
    print("Using TPV:", tpv_path)

    cap_fpv = cv2.VideoCapture(str(fpv_path))
    cap_tpv = cv2.VideoCapture(str(tpv_path))
    if not cap_fpv.isOpened() or not cap_tpv.isOpened():
        print("Failed to open FPV or TPV video")
        return []

    fps_fpv = cap_fpv.get(cv2.CAP_PROP_FPS) or 30.0
    fps_tpv = cap_tpv.get(cv2.CAP_PROP_FPS) or 30.0
    n_fpv = int(cap_fpv.get(cv2.CAP_PROP_FRAME_COUNT))
    n_tpv = int(cap_tpv.get(cv2.CAP_PROP_FRAME_COUNT))
    dur_fpv = n_fpv / fps_fpv if fps_fpv > 0 else 0.0
    dur_tpv = n_tpv / fps_tpv if fps_tpv > 0 else 0.0
    duration = min(dur_fpv, dur_tpv)
    print(f"FPV: fps={fps_fpv:.2f}, frames={n_fpv}, dur={dur_fpv:.2f}s")
    print(f"TPV: fps={fps_tpv:.2f}, frames={n_tpv}, dur={dur_tpv:.2f}s")

    seq: List[Dict] = []
    step = 1.0 / fps
    t = 0.0
    sampled = 0
    while t <= duration:
        # Seek FPV and TPV to time t (ms)
        cap_fpv.set(cv2.CAP_PROP_POS_MSEC, t * 1000.0)
        cap_tpv.set(cv2.CAP_PROP_POS_MSEC, t * 1000.0)
        ok_fpv, frame_fpv = cap_fpv.read()
        ok_tpv, frame_tpv = cap_tpv.read()
        if not ok_fpv or not ok_tpv:
            break

        counts_fpv, dets_fpv = yolo_counts_from_frame(frame_fpv, conf=conf)
        counts_tpv, dets_tpv = yolo_counts_from_frame(frame_tpv, conf=conf)

        fused_counts = Counter(counts_fpv)
        for k, v in counts_tpv.items():
            fused_counts[k] += v

        action = assign_action_label(t, actions)
        seq.append({
            "time_sec": t,
            "counts_fpv": dict(counts_fpv),
            "counts_tpv": dict(counts_tpv),
            "counts_fused": dict(fused_counts),
            "dets_fpv": dets_fpv,
            "dets_tpv": dets_tpv,
            "action": action,
        })

        sampled += 1
        if max_frames is not None and sampled >= max_frames:
            break
        t += step

    cap_fpv.release()
    cap_tpv.release()
    print(f"Built multiview sequence of length {len(seq)} for {instance_id}")
    return seq

In [91]:
# Turn fused YOLO counts into feature vectors + step/atomic labels

import numpy as np

# Use FineBio class names from the COCO categories / YOLO training
finebio_class_list = finebio_class_names  # length 35
name_to_index = {name: i for i, name in enumerate(finebio_class_list)}
feature_dim = len(finebio_class_list)
print("Feature dim (num FineBio classes):", feature_dim)


def counts_to_vector(counts: Dict[str, int]) -> np.ndarray:
    v = np.zeros(feature_dim, dtype=np.float32)
    for name, c in counts.items():
        idx = name_to_index.get(name)
        if idx is not None:
            v[idx] = float(c)
    return v


def multiview_sequence_to_arrays(seq: List[Dict]) -> Tuple[np.ndarray, np.ndarray]:
    """Use fused counts to build (X, y_task) for one sequence.

    - X: (T, D) fused YOLO counts per time step
    - y: (T,) integer step labels from `action.task` (-1 if no label)
    """
    T = len(seq)
    X = np.zeros((T, feature_dim), dtype=np.float32)
    y = np.full((T,), -1, dtype=np.int64)

    task_to_id: Dict[str, int] = {}
    next_id = 0

    for t, step in enumerate(seq):
        X[t] = counts_to_vector(step["counts_fused"])
        a = step["action"]
        if a and a.task:
            if a.task not in task_to_id:
                task_to_id[a.task] = next_id
                next_id += 1
            y[t] = task_to_id[a.task]

    print("Discovered", len(task_to_id), "distinct step tasks in this sequence")
    return X, y


# Example: build multiview sequence + arrays for one instance (no training yet)

mv_seq = build_multiview_sequence_for_instance("P03_02_01", fps=1.0, max_frames=20, conf=0.25)
print("Multiview sequence length:", len(mv_seq))
if mv_seq:
    X_mv, y_mv = multiview_sequence_to_arrays(mv_seq)
    print("X_mv shape:", X_mv.shape, "y_mv shape:", y_mv.shape)
    print("First 5 labels:", y_mv[:5])

Feature dim (num FineBio classes): 35
Loaded 169 actions for P03_02_01
Using FPV: /home/nyan/FineBio/04 finebio_videos_fpv_test/P03_02_01.mp4
Using TPV: /home/nyan/FineBio/08 finebio_videos_tpv_test/finebio_videos/P03_02_01_T1.mp4
FPV: fps=29.97, frames=4613, dur=153.92s
TPV: fps=29.97, frames=4613, dur=153.92s
Built multiview sequence of length 20 for P03_02_01
Multiview sequence length: 20
Discovered 1 distinct step tasks in this sequence
X_mv shape: (20, 35) y_mv shape: (20,)
First 5 labels: [-1  0  0  0  0]


## Build a multi-instance dataset for step recognition

Here we aggregate **multiview YOLO-26 sequences for all FineBio instances** and
build a dataset `(X, y)` suitable for training a temporal model to predict **steps**.

In [92]:
# Collect multiview sequences and build global step labels

from tqdm.auto import tqdm


def build_multiview_dataset(fps: float = 1.0,
                            max_frames: int = 60,
                            conf: float = 0.25):
    """Build a dataset of multiview sequences across all annotated instances.

    Returns:
        X_list: list of (T_i, D) numpy arrays of fused features
        y_list: list of (T_i,) int arrays of step labels (-1 for unlabeled frames)
        seq_instance_ids: list of instance_id strings
        task_to_id: dict mapping step task string -> label index
    """
    # 1) Collect sequences and discover all step task strings
    ds_sequences: List[Tuple[str, List[Dict]]] = []
    all_tasks = set()

    txt_files = sorted(ACTION_DIR.glob("P*.txt"))
    print(f"Found {len(txt_files)} action files")

    for txt_path in tqdm(txt_files, desc="Building multiview sequences"):
        instance_id = txt_path.stem
        try:
            mv_seq = build_multiview_sequence_for_instance(instance_id,
                                                           fps=fps,
                                                           max_frames=max_frames,
                                                           conf=conf)
        except Exception as e:
            print(f"Skipping {instance_id} due to error: {e}")
            continue
        if not mv_seq:
            continue
        for step in mv_seq:
            a = step["action"]
            if a and a.task:
                all_tasks.add(a.task)
        ds_sequences.append((instance_id, mv_seq))

    print(f"Collected {len(ds_sequences)} sequences")
    print(f"Discovered {len(all_tasks)} distinct step tasks")
    task_to_id = {task: i for i, task in enumerate(sorted(all_tasks))}

    # 2) Convert each sequence to (X_i, y_i)
    X_list: List[np.ndarray] = []
    y_list: List[np.ndarray] = []
    seq_instance_ids: List[str] = []

    for instance_id, mv_seq in ds_sequences:
        T = len(mv_seq)
        X = np.zeros((T, feature_dim), dtype=np.float32)
        y = np.full((T,), -1, dtype=np.int64)
        for t, step in enumerate(mv_seq):
            X[t] = counts_to_vector(step["counts_fused"])
            a = step["action"]
            if a and a.task and a.task in task_to_id:
                y[t] = task_to_id[a.task]
        if (y != -1).any():  # keep only sequences with at least one labeled frame
            X_list.append(X)
            y_list.append(y)
            seq_instance_ids.append(instance_id)

    print(f"Prepared {len(X_list)} sequences with at least one labeled frame")
    return X_list, y_list, seq_instance_ids, task_to_id


# Build the dataset (you can adjust fps/max_frames later if needed)
X_list, y_list, seq_instance_ids, task_to_id = build_multiview_dataset(
    fps=1.0,
    max_frames=120,
    conf=0.25,
)

print("Example sequence shapes:", [x.shape for x in X_list[:3]])
print("Number of unique step labels:", len(task_to_id))

/home/nyan/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Found 226 action files


Building multiview sequences:   0%|          | 0/226 [00:00<?, ?it/s]

Loaded 129 actions for P01_01_01
Using FPV: /home/nyan/FineBio/09 finebio_videos_fpv_train/finebio_videos/P01_01_01.mp4
Using TPV: /home/nyan/FineBio/05 finebio_videos_tpv_train/finebio_videos/P01_01_01_T1.mp4
FPV: fps=29.97, frames=4339, dur=144.78s
TPV: fps=29.97, frames=4339, dur=144.78s


Building multiview sequences:   0%|          | 1/226 [00:28<1:48:34, 28.95s/it]

...

Building multiview sequences: 100%|█████████▉| 225/226 [1:45:47<00:28, 28.29s/it]

Built multiview sequence of length 120 for P32_06_01
Loaded 307 actions for P32_07_01
Using FPV: /home/nyan/FineBio/11 finebio_videos_fpv_valid/finebio_videos/P32_07_01.mp4
Using TPV: /home/nyan/FineBio/06 finebio_videos_tpv_valid/finebio_videos/P32_07_01_T1.mp4
FPV: fps=29.97, frames=9471, dur=316.02s
TPV: fps=29.97, frames=9471, dur=316.02s


Building multiview sequences: 100%|██████████| 226/226 [1:46:15<00:00, 28.21s/it]

Built multiview sequence of length 120 for P32_07_01
Collected 226 sequences
Discovered 23 distinct step tasks
Prepared 226 sequences with at least one labeled frame


Example sequence shapes: [(120, 35), (120, 35), (120, 35)]
Number of unique step labels: 23


In [94]:
# PyTorch dataset + training loop for step recognition on GPU

from torch.utils.data import Dataset, DataLoader


class StepSequenceDataset(Dataset):
    def __init__(self, X_list: List[np.ndarray], y_list: List[np.ndarray]):
        assert len(X_list) == len(y_list)
        self.X_list = X_list
        self.y_list = y_list

    def __len__(self):
        return len(self.X_list)

    def __getitem__(self, idx):
        return self.X_list[idx], self.y_list[idx]


def collate_sequences(batch):
    """Pad variable-length sequences in a batch.

    Returns:
        X_pad: (B, T_max, D)
        y_pad: (B, T_max) with -1 for unlabeled / padded frames
        mask: (B, T_max) bool mask for labeled frames
    """
    lengths = [x.shape[0] for x, _ in batch]
    B = len(batch)
    T_max = max(lengths)

    X_pad = torch.zeros(B, T_max, feature_dim, dtype=torch.float32)
    y_pad = torch.full((B, T_max), -1, dtype=torch.long)
    mask = torch.zeros(B, T_max, dtype=torch.bool)

    for i, (x, y) in enumerate(batch):
        L = x.shape[0]
        X_pad[i, :L] = torch.from_numpy(x)
        y_torch = torch.from_numpy(y)
        y_pad[i, :L] = y_torch
        mask[i, :L] = (y_torch != -1)

    return X_pad, y_pad, mask


# Simple train/val split by sequence
num_sequences = len(X_list)
indices = np.arange(num_sequences)
np.random.seed(0)
np.random.shuffle(indices)

split = int(0.8 * num_sequences)
train_idx = indices[:split]
val_idx = indices[split:]

X_train = [X_list[i] for i in train_idx]
Y_train = [y_list[i] for i in train_idx]
X_val = [X_list[i] for i in val_idx]
Y_val = [y_list[i] for i in val_idx]

train_ds = StepSequenceDataset(X_train, Y_train)
val_ds = StepSequenceDataset(X_val, Y_val)

train_loader = DataLoader(train_ds, batch_size=4, shuffle=True, collate_fn=collate_sequences)
val_loader = DataLoader(val_ds, batch_size=4, shuffle=False, collate_fn=collate_sequences)


# Instantiate model for step recognition
num_step_classes = len(task_to_id)
input_dim = feature_dim
hidden_dim = 128

step_model = SimpleGRUHead(input_dim=input_dim, hidden_dim=hidden_dim, num_classes=num_step_classes)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
step_model.to(device)

criterion = nn.CrossEntropyLoss(ignore_index=-1)
optimizer = optim.Adam(step_model.parameters(), lr=1e-3)


def evaluate_step_model(model, loader):
    model.eval()
    correct = 0
    total = 0
    with torch.inference_mode():
        for Xb, yb, mask in loader:
            Xb = Xb.to(device)
            yb = yb.to(device)
            mask = mask.to(device)
            logits = model(Xb)  # (B, T, C)
            preds = logits.argmax(dim=-1)
            # Only consider frames with labels (mask True)
            correct += ((preds == yb) & mask).sum().item()
            total += mask.sum().item()
    acc = correct / total if total > 0 else 0.0
    return acc


# Training loop
num_epochs = 10

for epoch in range(1, num_epochs + 1):
    step_model.train()
    total_loss = 0.0
    n_batches = 0
    for Xb, yb, mask in train_loader:
        Xb = Xb.to(device)
        yb = yb.to(device)
        mask = mask.to(device)

        logits = step_model(Xb)  # (B, T, C)
        B, T, C = logits.shape
        loss = criterion(logits.view(B * T, C), yb.view(B * T))

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        n_batches += 1

    train_loss = total_loss / max(n_batches, 1)
    val_acc = evaluate_step_model(step_model, val_loader)
    print(f"Epoch {epoch}/{num_epochs} - train_loss={train_loss:.4f}, val_step_acc={val_acc:.3f}")

print("Done training step recognition model.")

Epoch 1/10 - train_loss=2.8438, val_step_acc=0.197
Epoch 2/10 - train_loss=2.4159, val_step_acc=0.191
Epoch 3/10 - train_loss=2.0481, val_step_acc=0.306
Epoch 4/10 - train_loss=1.9542, val_step_acc=0.291
Epoch 5/10 - train_loss=1.8256, val_step_acc=0.393
Epoch 6/10 - train_loss=1.6391, val_step_acc=0.415
Epoch 7/10 - train_loss=1.6224, val_step_acc=0.432
Epoch 8/10 - train_loss=1.5838, val_step_acc=0.425
Epoch 9/10 - train_loss=1.4577, val_step_acc=0.469
Epoch 10/10 - train_loss=1.3595, val_step_acc=0.479
Done training step recognition model.


## Diagnose low validation accuracy

47.9% accuracy on 23 step classes is low. Let's check:
1. Class distribution (imbalance?)
2. Label coverage (how many frames are actually labeled?)
3. Feature quality (are YOLO counts discriminative enough?)
4. Model predictions vs ground truth

In [ ]:
# Diagnose the low accuracy issue

# 1. Check class distribution
from collections import Counter

all_labels = []
for y in y_list:
    labeled = y[y != -1]
    all_labels.extend(labeled.tolist())

label_counts = Counter(all_labels)
print("Step class distribution:")
for label_id, count in sorted(label_counts.items()):
    task_name = [k for k, v in task_to_id.items() if v == label_id][0]
    print(f"  {label_id:2d} ({task_name:30s}): {count:5d} frames ({100*count/len(all_labels):.1f}%)")

print(f"\nTotal labeled frames: {len(all_labels)}")
print(f"Most common class: {label_counts.most_common(1)[0][1]} frames ({100*label_counts.most_common(1)[0][1]/len(all_labels):.1f}%)")
print(f"Random baseline (23 classes): {100/23:.1f}%")

# 2. Check label coverage per sequence
labeled_ratios = []
for y in y_list:
    ratio = (y != -1).sum() / len(y)
    labeled_ratios.append(ratio)

print(f"\nLabel coverage per sequence:")
print(f"  Mean: {np.mean(labeled_ratios):.2%}")
print(f"  Min: {np.min(labeled_ratios):.2%}")
print(f"  Max: {np.max(labeled_ratios):.2%}")

# 3. Check feature statistics
all_features = np.vstack(X_list)
print(f"\nFeature statistics:")
print(f"  Mean non-zero features per frame: {(all_features > 0).sum(axis=1).mean():.1f}")
print(f"  Max count per class: {all_features.max():.0f}")
print(f"  Mean count per class: {all_features[all_features > 0].mean():.2f}")

# 4. Check validation predictions
step_model.eval()
val_preds = []
val_labels = []
with torch.inference_mode():
    for Xb, yb, mask in val_loader:
        Xb = Xb.to(device)
        logits = step_model(Xb)
        preds = logits.argmax(dim=-1).cpu()
        for i in range(Xb.shape[0]):
            valid = mask[i].cpu()
            val_preds.extend(preds[i][valid].numpy())
            val_labels.extend(yb[i][valid].cpu().numpy())

from sklearn.metrics import confusion_matrix, classification_report

print("\nValidation confusion matrix (top 10 classes by frequency):")
top_classes = sorted(label_counts.items(), key=lambda x: x[1], reverse=True)[:10]
top_class_ids = [c[0] for c in top_classes]
val_preds_arr = np.array(val_preds)
val_labels_arr = np.array(val_labels)

# Filter to top classes only
mask_top = np.isin(val_labels_arr, top_class_ids)
if mask_top.sum() > 0:
    cm = confusion_matrix(val_labels_arr[mask_top], val_preds_arr[mask_top], labels=top_class_ids)
    print("Rows=ground truth, Cols=predicted")
    for i, label_id in enumerate(top_class_ids):
        task_name = [k for k, v in task_to_id.items() if v == label_id][0]
        print(f"  {label_id:2d} {task_name[:25]:25s}: {cm[i].sum():4d} true, pred={top_class_ids[cm[i].argmax()]:2d} (acc={100*cm[i,i]/max(cm[i].sum(),1):.1f}%)")